# Lab: Efficient Storage and Serialisation in Python

This lab uses the **NYC TLC Yellow Taxi** trip records as a realistic tabular dataset.

The focus is: "Which format is best for my data?"

Main goals:

- compare text and binary storage
- compare row style and column style thinking
- use `CSV`, `Pickle`, `Parquet`, `Feather`, and NumPy `NPZ`
- inspect Arrow schemas
- measure file size, write time, and read time
- decide which format fits which task

## 1) Setup

For this lab, you may need:

```bash
pip install pandas numpy
```

In [4]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

### 1.1) Download and load the dataset

This notebook uses one official monthly Yellow Taxi parquet file from the [NYC TLC data portal](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).

We use one monthly parquet file, then keep only a manageable sample for the lab.

You can change the URL to another month later.

In [5]:
DATA_DIR = Path("storage_lab_data")
DATA_DIR.mkdir(exist_ok=True)

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2022-01.parquet"
parquet_source = DATA_DIR / "yellow_tripdata_2022_01.parquet"

if not parquet_source.exists():
    df_full = pd.read_parquet(url)
    df_full.to_parquet(parquet_source, index=False)
else:
    df_full = pd.read_parquet(parquet_source)

print(f"Full shape: {df_full.shape}")
df_full.head()

Full shape: (2463931, 19)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2022-01-01 00:35:40,2022-01-01 00:53:29,2.0,3.80,1.0,N,142,236,1,14.5,3.0,0.5,3.65,0.0,0.3,21.95,2.5,0.0
1,1,2022-01-01 00:33:43,2022-01-01 00:42:07,1.0,2.10,1.0,N,236,42,1,8.0,0.5,0.5,4.00,0.0,0.3,13.30,0.0,0.0
2,2,2022-01-01 00:53:21,2022-01-01 01:02:19,1.0,0.97,1.0,N,166,166,1,7.5,0.5,0.5,1.76,0.0,0.3,10.56,0.0,0.0
3,2,2022-01-01 00:25:21,2022-01-01 00:35:23,1.0,1.09,1.0,N,114,68,2,8.0,0.5,0.5,0.00,0.0,0.3,11.80,2.5,0.0
4,2,2022-01-01 00:36:48,2022-01-01 01:14:20,1.0,4.30,1.0,N,68,163,1,23.5,0.5,0.5,3.00,0.0,0.3,30.30,2.5,0.0


**Notice that the original file to download is in `parquet` format.**

💭 Why do you think this is the case? 🤔

### 1.2) Build a neat working subset

We keep only selected columns and a fixed sample size.
This keeps the lab fast and reproducible.

In [6]:
columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "total_amount",
]

df = df_full[columns].sample(n=100_000, random_state=42).reset_index(drop=True).copy()

# Light cleanup for consistency
df["store_and_fwd_flag"] = df["store_and_fwd_flag"].astype("category")
df["payment_type"] = df["payment_type"].astype("Int64")
df["RatecodeID"] = df["RatecodeID"].astype("Int64")
df["PULocationID"] = df["PULocationID"].astype("Int64")
df["DOLocationID"] = df["DOLocationID"].astype("Int64")

print(df.shape)
df.head()

(100000, 15)


,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,total_amount
0,2022-01-15 00:22:49,2022-01-15 00:31:33,2.0,1.50,1,N,148,209,1,8.0,3.0,0.5,2.36,0.0,14.16
1,2022-01-13 17:50:32,2022-01-13 18:03:09,1.0,2.60,1,N,48,238,1,11.0,3.5,0.5,1.00,0.0,16.30
2,2022-01-19 18:41:39,2022-01-19 18:56:29,1.0,3.73,1,N,141,24,1,13.5,1.0,0.5,3.56,0.0,21.36
3,2022-01-08 10:35:44,2022-01-08 10:45:53,1.0,5.96,1,N,231,140,1,17.5,0.0,0.5,4.16,0.0,24.96
4,2022-01-26 20:08:13,2022-01-26 20:12:15,2.0,0.69,1,N,229,162,2,5.0,0.5,0.5,0.00,0.0,8.80


### 1.3) Quick `dtype` inspection

Storage format choice becomes much easier when you look at data types first.

In [4]:
df.dtypes

tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                        Int64
store_and_fwd_flag             category
PULocationID                      Int64
DOLocationID                      Int64
payment_type                      Int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
total_amount                    float64
dtype: object

## 2) Exercises

### 2.0) Helper functions for benchmarking

We measure:

- Write time
- Read time
- File size

Implement functions that measure execution time and file size.

You are expected to use these functions for every exercise in this section, and compare the results afterwards.

### 2.1) NPY and NPZ

Save the `df` in `.npy` and/or `.npz` format 🤔👀

### 2.2) CSV
Save the `df` as `.csv`.

### 2.3) Pickle

"Pickle" the `df`.

### 2.4) Parquet
Save the `df` as `.parquet`.

### 2.5) Feather
Save the `df` as `.feather`.


### 2.6) Compare results

Compare all the data you gathered above and plot the results.

### 2.7) Optional extension

Try one of these:

- compare `np.save` with `np.savez_compressed`
- test another sample size
- test another month of NYC taxi data
- add memory profiling with `df.memory_usage(deep=True)`

## 3) Observation and Discussion

### 3.1) Check dtype preservation

Formats differ in how well they preserve structure and types. Check the data type of the data you saved.

### 3.2) Parquet column selection demo

`Parquet` is especially well suited to columnar storage and column wise reading. Read a few columns from your saved Parquet data while tracking the read time.

### 3.3) Take-home lessons

Fill in this section based on your observations and understanding of the topic.